# Neural networks
for Body mass prediction in biped dinosaurs using supervised learning
-----------------------------------------------------------------------------------------------------------------------------
Alejandro Otero - CONICET - Museo de La Plata (UNLP), Argentina. alexandros.otero@gmail.com

Ailen Borya - Especialización en Geociencias de Exploración y Desarrollo de Hidrocarburos (UNLP), Argentina. ailenborya@gmail.com

## 1. Libraries

In [ ]:
# Libraries

!pip install scikeras
!pip install keras-tuner
!pip install shap
import shap
import tensorflow as tf
import keras
import keras.backend as k
from keras import layers
import keras_tuner as kt
import os
import math
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits import mplot3d
import numpy as np
import seaborn as sns
#from plotly.offline import plot
#import plotly.express as px
#import plotly.graph_objects as go
#import plotly.io as pio
from sklearn import preprocessing
from keras import regularizers
#from sklearn.neural_network import MLPRegressor
#from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn import metrics
from six import StringIO
from sklearn.tree import export_graphviz
#import pydotplus
#from pydotplus import graph_from_dot_data
from sklearn.datasets import make_blobs
#from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
#from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model

## 2. Open file

In [ ]:
# Maximum columns to display

pd.options.display.max_columns = 100

In [ ]:
# Suppress scientific notation by setting float_format

pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Set random seed for reproductivility

keras.utils.set_random_seed(42)

In [ ]:
# Directory location

os.getcwd()

In [ ]:
# Colab open file
'''
from google.colab import files
uploaded = files.upload()
'''

In [ ]:
# Open file

df = pd.read_excel('train_data.xlsx')
df.head(5) # Visualising first 5 rows

## 3. Data split

In [ ]:
# Store predictor variables in X
# Store target variable in y

X = df.filter([
    'Femur length (mm)',
    'Femur circumference (mm)'
])
y = df['Body mass (g)']

In [ ]:
# Train and Test split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.25, # split ratio 72:25
    random_state = 42 # enables replication
)

## 4. Preprocessing

In [ ]:
# Convert all numerical variables to natural logarithm

# log-transform predictors
X_train_log = np.log(X_train)
X_test_log  = np.log(X_test)

# log-transform target
y_train_log = np.log(y_train)
y_test_log  = np.log(y_test)

In [ ]:
# Normalization via StandardScaler

X_scaler = StandardScaler()

X_train_s = X_scaler.fit_transform(X_train_log)
X_test_s  = X_scaler.transform(X_test_log)

In [ ]:
# Convert target to dataframe and normalized

y_scaler = StandardScaler()

y_train_s = y_scaler.fit_transform(y_train_log.values.reshape(-1, 1)).ravel()
y_test_s = y_scaler.transform(y_test_log.values.reshape(-1, 1)).ravel()

## 5. Model creation and fit

In [ ]:
# Define the Sequential model
# Experiment with deeper and wider networks

# Clear the session to enable re-run the model several times
k.clear_session()

model = keras.Sequential()

# Input shape equls the variable dimensions of the dataset
model.add(Input(shape = (2,))) # input = 1 for one predictor; input = 2 for two predictors
#model.add(layers.BatchNormalization())
'''
# Internal layers plain
model.add(layers.Dense(20, activation = 'relu'))
model.add(layers.Dense(20, activation = 'relu'))
'''
# Internal layers with L2 regularization
model.add(layers.Dense(20, kernel_regularizer = regularizers.l2(0.001), activation = 'relu'))
model.add(layers.Dense(20, kernel_regularizer = regularizers.l2(0.001), activation = 'relu'))
'''
# Internal layers with Batch Normalization
model.add(layers.Dense(20, use_bias = True))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dense(20, use_bias = True))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

# Internal layers with Dropout
model.add(layers.Dense(20, activation = 'relu'))
model.add(layers.Dropout(0.1, seed = 42))
model.add(layers.Dense(20, activation = 'relu'))
model.add(layers.Dropout(0.1, seed = 42))
'''

# Output layer expecting 1 output
#model.add(layers.Dense(1, activation ='linear'))
model.add(layers.Dense(1))

In [ ]:
# Summary of the parameters of the model

model.summary()

In [ ]:
# Model Compile

optimizer = keras.optimizers.Adam(learning_rate = 0.001)

model.compile(
    loss = 'mean_squared_error',
    optimizer = optimizer,
    metrics = ['mae', 'mse', 'r2_score']
)

In [ ]:
# Model training
# It is important to store the fitting into a variable in order to visualize the history after.

history = model.fit(
    X_train_s,
    y_train_s,
    validation_data = (X_test_s, y_test_s),
    validation_split = 0.25,
    epochs = 100
)

## 5. Learning curves

In [ ]:
# Loss

loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.figure(figsize = (6, 4))
plt.plot(epochs, loss, 'y', label = 'Training loss')
plt.plot(epochs, val_loss, 'r', label = 'Testing loss')
plt.title('Training and testing loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
#plt.savefig('.jpg', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
# MSE

mse = history.history['mse']
val_mse = history.history['val_mse']
plt.figure(figsize = (6, 4))
plt.plot(epochs, mse, 'y', label ='Training MSE')
plt.plot(epochs, val_mse, 'r', label ='Testing MSE')
plt.title('Training and testing MSE')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.legend()
#plt.savefig('MSE_train_extant.jpg', dpi = 500, bbox_inches = 'tight')
plt.show()

## 6. Model prediction and performance

In [ ]:
# Prediction on test data

y_pred_test = model.predict(X_test_s)

# Retrotransformation
# back to log(BM)

y_pred_log = y_scaler.inverse_transform(
    y_pred_test.reshape(-1, 1)).ravel()

# back to BM
y_pred = np.exp(y_pred_log)
y_true = np.exp(y_test_log) # the same as y_test

In [ ]:
# Model prediction and performance

# Mean Squared Error (MSE)
# It calculates the average of the squares of the errors, which gives a higher weight to larger errors.

mse_test = (mean_squared_error(y_true, y_pred))

# R-squared (Coefficient of Determination)
# It indicates the proportion of variance in the dependent variable that can be predicted from the independent variables.

r2_test = r2_score(y_true, y_pred)

print('Mean squared error MSE_test: ', mse_test)
print('\n')
print('R2_test: ', r2_test)

## 7. Predicting new data on validation set

In [ ]:
# Open validation file

df_val = pd.read_excel('val_data.xlsx')
df_val.head()

In [ ]:
# Store predictor variables in X
# Store target variable in y

X_val = df_val.filter([
    'Femur length (mm)',
    'Femur circumference (mm)'
])
y_val = df_val['Body mass (g)']

In [ ]:
# Convert all numerical variables to natural logarithm

X_val_log = np.log(X_val)
y_val_log  = np.log(y_val)

In [ ]:
# Normalization (DO NOT REFIT)

X_val_s = X_scaler.transform(X_val_log)
y_val_s = y_scaler.transform(y_val_log.values.reshape(-1, 1)).ravel()

In [ ]:
# Prediction new value

df_val_pred = model.predict(X_val_s)

In [ ]:
# Undo standarization - back to Log

y_new_log = y_scaler.inverse_transform(
   df_val_pred.reshape(-1, 1)).ravel()

In [ ]:
# Retrotransform - back to BM

y_val_pred = np.exp(y_new_log) # predicted values (in grams)
y_val_true = np.exp(y_val_log) # real values (in grams) ---> the original y_val

## 8. Assessing performance on validation set

In [ ]:
# Model prediction and performance

# Mean Squared Error (MSE)
# It calculates the average of the squares of the errors, which gives a higher weight to larger errors.

mse_val = (mean_squared_error(y_val_true, y_val_pred))

# R-squared (Coefficient of Determination)
# It indicates the proportion of variance in the dependent variable that can be predicted from the independent variables.

r2_val = r2_score(y_val_true, y_val_pred)

print('Mean squared error MSE_val: ', mse_val)
print('\n')
print('R2_val: ', r2_val)

In [ ]:
## Ratio estimator (Snowdon, 1991)
# mean sample original / mean back-transformed predicted values

RE = y_val.mean() / y_val_pred.mean()
y_val_pred_RE = y_val_pred*RE
RE

In [ ]:
# Convert predictions to dataframe

y_val_pred_RE = pd.DataFrame(y_val_pred_RE)
y_val_pred_RE.head()

In [ ]:
# Save predictions in excel file

y_val_pred_RE.to_excel('val_pred_nn.xlsx', float_format = '%.2f')

## 9. Feature importance on validation set

In [ ]:
# SHAP

# Convert scaled arrays back to DataFrames with column names
feature_names = ["Femur length (mm)", "Femur circumference (mm)"]

X_train_df = pd.DataFrame(X_train_s, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_s, columns=feature_names)

# Use TreeExplainer for tree-based models
explainer_val = shap.Explainer(model, X_train_df) # can use also X_train

# Calculate SHAP values for a subset (e.g. test set)
# The predictions are the output of y_pred_test
shap_values_val = explainer_val(X_val_s) # can use also X_test

# Summary plot (bar chart of average absolute SHAP values)
#shap.plots.bar(shap_values)

# Compute mean absolute SHAP values
mean_abs_shap_val = np.abs(shap_values_val.values).mean(axis = 0)

# Create a DataFrame for sorting
importance_df_val = pd.DataFrame({
    'feature': feature_names,
    'importance': mean_abs_shap_val
}).sort_values('importance', ascending = True)

# Custom bar plot with green color
plt.figure(figsize = (6, 2.5))
plt.barh(importance_df_val['feature'], importance_df_val['importance'], color = 'red')
plt.xlabel("Mean |SHAP value|")
plt.title("Neural networks SHAP (validation set)")
plt.tight_layout()
#plt.savefig('nn_shap_val_extant.jpg', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
fig = shap.plots.beeswarm(
    shap_values_val, 
    show = False,
    color = plt.get_cmap('jet'))

#plt.savefig('nn_shap2_val_extant.jpg', dpi = 500, bbox_inches = 'tight')

plt.show()

In [ ]:
shap.plots.initjs()

In [ ]:
# visualize the first prediction's explanation with a force plot
shap.plots.force(shap_values_val[7])
#plt.title("RFdh SHAP force")
#plt.savefig('RFdh_force.jpg', dpi = 500, bbox_inches = 'tight')

## 10. Predicting biped dinosaurs (dataset from Campione et al., 2014, table 3)

IMPORTANT: Using only femoral circumference (FC) to compre with cQE. To do so, you need to train the sequential model with an input layer of 1.

In [ ]:
# Open  file

df_bip = pd.read_excel('val_bipeds.xlsx')
df_bip.head()

In [ ]:
# Removing non-numerical columns

df_bip = df_bip.iloc[:,[1]]
df_bip.head(5)

In [ ]:
# Convert all numerical variables to natural logarithm

df_bip_log = np.log(df_bip)
df_bip_log.head()

In [ ]:
# Normalization (DO NOT REFIT)

df_bip_s = X_scaler.transform(df_bip_log)

In [ ]:
# Prediction new value

df_bip_pred = model.predict(df_bip_s)

In [ ]:
# Undo standarization - back to Log

y_bip_log = y_scaler.inverse_transform(
   df_bip_pred.reshape(-1, 1)).ravel()

In [ ]:
# Retrotransform - back to BM

y_bip_pred = np.exp(y_bip_log) # predicted values (in grams)

In [ ]:
# Convert predictions to dataframe

df_bip_pred = pd.DataFrame(y_bip_pred)
df_bip_pred.head()

## End code